# CudaSIFT + RootSIFT (GPU) ORB-SLAM3 fork -- KITTI whole-set sweep (Kaggle GPU)

Runs `orbslam3_sift_kitti_ate` (the SIFT-feature fork of ORB-SLAM3, GPU-accelerated at
the extractor via [Celebrandil/CudaSift](https://github.com/Celebrandil/CudaSift) +
RootSIFT) against KITTI odometry sequences 00-05, sweeping runtime toggles for the
tracker (KLT vs SQPnP) and the epipolar-bridge recovery mechanism (which keeps a map
alive across a tracking dropout instead of resetting -- see
`ORBSLAM3_SIFT_EPIPOLAR_BRIDGE_PLAN.md` in the repo for the full writeup).

**Before running:**
1. Notebook settings -> **Accelerator: GPU** (T4 x2 or P100). **Internet: On**.
2. **Add Data** -> attach a KITTI odometry (grayscale) Kaggle Dataset, e.g.
   `xuehu12/kitti-odometry-gray` (images only -- ground-truth poses come from this
   repo's own `kitti_poses/`, not the dataset; `kaggle/setup_and_run.sh` handles that
   fallback automatically).
3. `REPO_URL` below must point at your pushed repo (public, or bake a token into the
   URL) -- the commits this sweep depends on (epipolar bridge, SQPnP tracker,
   mono-init stale-reference fix, the `TRACKER`/`BRIDGE`/`INITFIX` runtime toggles,
   and the dataset auto-detect fix) must already be pushed to `main`.

**What the toggles are** (env vars, read once at startup, no rebuild needed to change
them -- see `Tracking::trackerUsesSqpnp()`/`bridgeMode()`/`initFixEnabled()`):
- `TRACKER=klt|sqpnp` -- per-frame tracker (default sqpnp).
- `BRIDGE=off|rl|lost|both` -- epipolar-bridge recovery: `rl` = only in the
  RECENTLY_LOST branch (established maps losing track), `lost` = only in the LOST
  branch (young maps, <10 KFs, that would otherwise reset immediately -- this is
  where most of Kaggle's first-run resets came from), `both` = try it in either
  (default).
- `INITFIX=on|off` -- advance the mono-init reference once matched disparity exceeds
  150px, instead of retrying forever against a frozen stale reference (fixes seq01's
  highway degenerate-init problem; default on).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader


In [ ]:
import os

REPO_URL = "https://github.com/nguyenhuunam852/ORB-SLAM-SIFT.git"  # set to your pushed repo
REPO_DIR = "/kaggle/working/ORB-SLAM-SIFT"

if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    print(f"{REPO_DIR} already present, skipping clone (rm -rf it to force a fresh clone)")


## Build only

One-time: apt deps, g2o build, CudaSift clone, ONNX Runtime GPU download, cmake
configure + build (`USE_CUDASIFT=1`). `BUILD_ONLY=1` stops right after the build --
the actual benchmark runs all happen in the sweep cell below, so this cell doesn't
waste GPU time on a redundant default run. Safe to rerun (skips steps whose output
already exists). If a later cell errors with *"Could not find a KITTI sequence
directory"*, the dataset isn't attached yet (see step 2 above), not a code problem.

In [ ]:
import os
os.environ["BUILD_ONLY"] = "1"  # build the toolchain only -- the sweep cell below runs the actual benchmarks
!cd {REPO_DIR} && bash kaggle/setup_and_run.sh


## Main sweep: 6 configs x sequences 00-05

Pure bash (`%%bash` cell magic) -- no Python string-interpolation of `{REPO_DIR}`
involved. `SKIP_BUILD=1` reuses the build above -- no rebuild between configs, since
`TRACKER`/`BRIDGE`/`INITFIX` are pure runtime env-var toggles. Edit `SEQS` or
`CONFIGS` inside the cell to narrow the sweep; each (config, seq) pair runs the full
sequence (`MAX_FRAMES=0`) and its stdout is captured to
`/kaggle/working/<config>_seq<NN>.log`. Results also land in
`/kaggle/working/sweep_results.tsv` (tab-separated) if you want to re-print or
re-sort the table without rerunning anything.

**Configs swept:**
- `baseline_klt_nobridge` -- pre-recipe behavior (KLT, no bridge, no init-fix): the
  number every other row needs to beat.
- `klt_bridge_both` -- isolates the bridge's effect on top of the original KLT
  tracker.
- `sqpnp_nobridge` -- isolates SQPnP's own effect on seq00 (an open question from the
  first Kaggle run: is high reset count from SQPnP itself, or from the bridge being
  bypassed by young-map deaths?).
- `sqpnp_bridge_rl` / `sqpnp_bridge_lost` / `sqpnp_bridge_both` -- isolates which
  bridge branch (established-map loss vs young-map rescue) actually helps.

In [ ]:
%%bash
set -uo pipefail

REPO_DIR="/kaggle/working/ORB-SLAM-SIFT"
SEQS="00 01 02 03 04 05"

# name:tracker:bridge:initfix
CONFIGS="
baseline_klt_nobridge:klt:off:off
klt_bridge_both:klt:both:on
sqpnp_nobridge:sqpnp:off:on
sqpnp_bridge_rl:sqpnp:rl:on
sqpnp_bridge_lost:sqpnp:lost:on
sqpnp_bridge_both:sqpnp:both:on
"

SEQ_BASE="$(find /kaggle/input -maxdepth 8 -type d -name sequences 2>/dev/null | sort | head -1)"
[ -z "${SEQ_BASE}" ] && SEQ_BASE="/kaggle/input/kitti-odometry-gray/dataset/sequences"
echo "SEQ_BASE=${SEQ_BASE}  (edit this script's SEQ_BASE line if wrong)"

# Ground-truth total path length per sequence, computed directly from
# kitti_poses/<seq>.txt (NOT parsed from the harness log -- orbslam3_kitti_ate.cpp
# never prints a "GT path length" line; that string only exists in the OTHER
# harness, kitti_ate.cpp, for the custom SlamWorker pipeline. Sum of consecutive
# (tx,tz) translation deltas, columns 4 and 12 of the flattened 12-value pose).
gt_path_length() {
    awk '{
        x=$4; z=$12
        if (NR>1) { dx=x-px; dz=z-pz; s+=sqrt(dx*dx+dz*dz) }
        px=x; pz=z
    } END{printf "%.1f", s+0}' "${REPO_DIR}/kitti_poses/${1}.txt"
}

RESULTS="/kaggle/working/sweep_results.tsv"
: > "${RESULTS}"

for entry in ${CONFIGS}; do
    cfg_name="${entry%%:*}"; rest="${entry#*:}"
    tracker="${rest%%:*}"; rest="${rest#*:}"
    bridge="${rest%%:*}"; initfix="${rest#*:}"

    for seq in ${SEQS}; do
        seq_dir="${SEQ_BASE}/${seq}"
        poses="${REPO_DIR}/kitti_poses/${seq}.txt"
        if [ ! -d "${seq_dir}/image_0" ] || [ ! -f "${poses}" ]; then
            echo "[skip ${cfg_name}/${seq}] missing images or poses"
            continue
        fi
        gt_m="$(gt_path_length "${seq}")"
        logf="/kaggle/working/${cfg_name}_seq${seq}.log"
        echo "=== ${cfg_name} / seq${seq} (TRACKER=${tracker} BRIDGE=${bridge} INITFIX=${initfix}, GT=${gt_m}m) ==="
        ( cd "${REPO_DIR}" && \
          SKIP_BUILD=1 USE_CUDASIFT=1 START_FRAME=0 MAX_FRAMES=0 \
          KITTI_SEQ_DIR="${seq_dir}" KITTI_POSES="${poses}" \
          OUT_PREFIX="/kaggle/working/${cfg_name}_seq${seq}" \
          TRACKER="${tracker}" BRIDGE="${bridge}" INITFIX="${initfix}" \
          bash kaggle/setup_and_run.sh ) > "${logf}" 2>&1

        cov_m=$(grep -oE 'pathLen=[0-9.]+m' "${logf}" | grep -oE '[0-9.]+' | awk '{s+=$1} END{printf "%.1f", s+0}')
        worst=$(grep -oE 'rmse/mean/median/max\)=[0-9./]+' "${logf}" | grep -oE '[0-9.]+$' | awk 'BEGIN{m=0} {if($1+0>m)m=$1+0} END{printf "%.2f", m}')
        [ -z "${worst}" ] && worst=0
        resets=$(grep -cE 'A new map is started|Reseting current map' "${logf}")
        bridge_ok=$(grep -cE '\[bridge\].*SUCCESS' "${logf}")
        ninit=$(grep -cE 'mono-init.*SUCCESS' "${logf}")
        nfail=$(grep -cE 'reconstruct FAILED' "${logf}")
        frags=$(grep -cE '^\[fragment ' "${logf}")
        covp=$(awk -v c="${cov_m}" -v g="${gt_m}" 'BEGIN{printf "%.1f", (g>0)?100.0*c/g:0}')
        initp=$(awk -v i="${ninit}" -v f="${nfail}" 'BEGIN{d=i+f; printf "%.1f", (d>0)?100.0*i/d:0}')

        printf "%s\t%s\t%s\t%s%%\t%sm\t%s\t%s\t%s%%\t%s\n" \
            "${cfg_name}" "${seq}" "${cov_m}" "${covp}" "${worst}" "${resets}" "${bridge_ok}" "${initp}" "${frags}" \
            >> "${RESULTS}"
        echo "    cov=${cov_m}m/${gt_m}m=${covp}%  worstATE=${worst}m  resets=${resets}  bridgeOK=${bridge_ok}  init%=${initp}  frags=${frags}"
    done
done

echo ""
echo ""
echo "=== RESULTS TABLE (toggle sweep, seq 00-05) ==="
echo ""
echo "| Config | Seq | Cov(m) | Cov% | worstATE | Resets | BridgeOK | Init% | Frags |"
echo "|--------|-----|--------|------|----------|--------|----------|-------|-------|"
awk -F'\t' '{printf "| %s | %s | %s | %s | %s | %s | %s | %s | %s |\n", $1,$2,$3,$4,$5,$6,$7,$8,$9}' "${RESULTS}"


## Reading the table

- **`baseline_klt_nobridge`** is the number every other row must beat -- coverage and
  reset count with none of this session's changes active.
- Compare **`sqpnp_nobridge`** against baseline to isolate whether SQPnP itself helps
  or hurts seq00 independent of the bridge.
- Compare **`sqpnp_bridge_rl`** vs **`sqpnp_bridge_lost`** vs **`sqpnp_bridge_both`** to
  see which bridge branch (established-map loss vs young-map rescue) is doing the
  work -- the first Kaggle run (no LOST-path bridge yet) fired only 45x against 317
  resets, because most maps died young (<10 KFs) and bypassed the RECENTLY_LOST branch
  entirely.
- Per-sequence logs are at `/kaggle/working/<config>_seq<NN>.log` if a row needs
  closer inspection (grep `[bridge]`, `[mono-init]`, `Fail to track local map`).